In [1]:
import pandas as pd

# The missing volumes

In [2]:
fpt = pd.read_parquet("../data/fpt_intraday_2026-11-06.parquet")
fpt_kbs = pd.read_parquet("../data/fpt_intraday_kbs_2026-11-06.parquet")

In [3]:
fpt_kbs["total_so_far"] = fpt_kbs["id"].str.split("_").str[1]
fpt_kbs

,time,price,volume,match_type,id,total_so_far
0,2026-11-06 09:40:23+07:00,51.5,2000,buy,20261106_3316900,3316900
1,2026-11-06 09:40:26+07:00,51.4,100,sell,20261106_3317000,3317000
2,2026-11-06 09:40:31+07:00,51.5,100,buy,20261106_3317100,3317100
3,2026-11-06 09:40:32+07:00,51.4,1000,sell,20261106_3318100,3318100
4,2026-11-06 09:40:32+07:00,51.5,1100,buy,20261106_3319200,3319200
...,...,...,...,...,...,...
4995,2026-11-06 14:29:51+07:00,51.0,300,buy,20261106_11378500,11378500
4996,2026-11-06 14:29:51+07:00,50.9,200,sell,20261106_11378700,11378700
4997,2026-11-06 14:29:54+07:00,50.9,100,sell,20261106_11378800,11378800
4998,2026-11-06 14:29:58+07:00,50.9,20000,sell,20261106_11398800,11398800


In [4]:
fpt = fpt.sort_values(by="time", ascending=True, kind="stable")
fpt["total_so_far"] = fpt["volume"].cumsum()

In [5]:
fpt

,time,price,volume,match_type,id,total_so_far
0,2026-11-06 09:15:00+07:00,51.4,600,Sell,534080834,600
1,2026-11-06 09:15:00+07:00,51.5,500,Buy,534080827,1100
2,2026-11-06 09:15:00+07:00,51.5,500,Buy,534080823,1600
3,2026-11-06 09:15:00+07:00,51.4,370500,ATO,534080562,372100
4,2026-11-06 09:15:01+07:00,51.5,200,Buy,534080931,372300
...,...,...,...,...,...,...
14609,2026-11-06 14:29:57+07:00,50.9,100,Sell,534624250,11396600
14610,2026-11-06 14:29:57+07:00,50.9,200,Sell,534624249,11396800
14611,2026-11-06 14:29:57+07:00,50.9,500,Sell,534624248,11397300
14612,2026-11-06 14:29:57+07:00,50.9,1400,Sell,534624247,11398700


In [6]:
fpt_kbs["minute"] = fpt_kbs["time"].dt.floor("min")
fpt["minute"] = fpt["time"].dt.floor("min")

In [7]:
fpt_kbs["total_so_far"] = fpt_kbs["total_so_far"].astype("int64")

fpt_kbs_minute = fpt_kbs.groupby("minute").agg({"total_so_far": "max"})
fpt_minute = fpt.groupby("minute").agg({"total_so_far": "max"})

In [8]:
merging_minute = pd.merge(fpt_kbs_minute,
                   fpt_minute,
                   on="minute",
                   how="outer")

In [9]:
merging_minute["vci_minus_kbs"] = merging_minute["total_so_far_y"] - merging_minute["total_so_far_x"]
merging_minute

,total_so_far_x,total_so_far_y,vci_minus_kbs
minute,,,
2026-11-06 09:15:00+07:00,NaN,601300,NaN
2026-11-06 09:16:00+07:00,NaN,772000,NaN
2026-11-06 09:17:00+07:00,NaN,964800,NaN
2026-11-06 09:18:00+07:00,NaN,1172300,NaN
2026-11-06 09:19:00+07:00,NaN,1222400,NaN
...,...,...,...
2026-11-06 14:26:00+07:00,11200400.0,11200300,-100.0
2026-11-06 14:27:00+07:00,11244800.0,11244700,-100.0
2026-11-06 14:28:00+07:00,11285000.0,11284900,-100.0


In [10]:
merging_minute[(merging_minute["vci_minus_kbs"].notna()) & (merging_minute["vci_minus_kbs"] != 0)]

,total_so_far_x,total_so_far_y,vci_minus_kbs
minute,,,
2026-11-06 09:48:00+07:00,3701800.0,3703800,2000.0
2026-11-06 10:14:00+07:00,4491100.0,4491200,100.0
2026-11-06 10:16:00+07:00,4509100.0,4509600,500.0
2026-11-06 10:20:00+07:00,4736900.0,4737100,200.0
2026-11-06 10:34:00+07:00,5069000.0,5069500,500.0
2026-11-06 10:46:00+07:00,5467800.0,5502800,35000.0
2026-11-06 10:52:00+07:00,5603600.0,5604600,1000.0
2026-11-06 13:00:00+07:00,6856800.0,6856900,100.0
2026-11-06 13:11:00+07:00,7314800.0,7315900,1100.0


- So at `2026-11-06 14:22:00+07:00` the difference stuck at -100.0

In [11]:
fpt_vci_missing_gap = fpt[(fpt["time"] >= "2026-11-06 14:21:30") & (fpt["time"] <= "2026-11-06 14:23:00")]
fpt_kbs_missing_gap = fpt_kbs[(fpt_kbs["time"] >= "2026-11-06 14:21:30") & (fpt_kbs["time"] <= "2026-11-06 14:23:00")]

In [12]:
missing_gap_compare = fpt_vci_missing_gap.groupby(["price", "volume"]).size() - fpt_kbs_missing_gap.groupby(["price", "volume"]).size()

In [13]:
missing_gap_compare = pd.DataFrame(missing_gap_compare)
missing_gap_compare = missing_gap_compare.rename(columns={0: "size"})

In [14]:
missing_gap_compare[missing_gap_compare["size"] != 0]

size
price volume      
50.9  100     40.0
      200     15.0
      300      7.0
      400      7.0
      500      NaN
      600      1.0
      700      NaN
      800      NaN
      1000     NaN
      1300     NaN
      1400     NaN
      1500     NaN
      2000     1.0
      2500     NaN
      4000     NaN
      5000     NaN
      7800     NaN
      8300     NaN
      8600     NaN
      10500    NaN
      20000    NaN
      20500    NaN
      30000    NaN
51.0  100      4.0
      200      5.0
      300      1.0
      400      NaN
      500      NaN
      600      NaN
      700      NaN
      800      1.0
      1000    -2.0
      1500     NaN
      2500     NaN
      2600     NaN
      3000     NaN
      5000     NaN
      9900     NaN

- Does not work: `vci` and `kbs` split trades into different rows

In [15]:
fpt_kbs_second = fpt_kbs_missing_gap.groupby("time").agg({"total_so_far": "max"})
fpt_second = fpt_vci_missing_gap.groupby("time").agg({"total_so_far": "max"})

In [16]:
merging_second = pd.merge(fpt_kbs_second,
                   fpt_second,
                   on="time",
                   how="outer")

In [17]:
merging_second["vci_minus_kbs"] = merging_second["total_so_far_y"] - merging_second["total_so_far_x"]
merging_second

,total_so_far_x,total_so_far_y,vci_minus_kbs
time,,,
2026-11-06 14:21:30+07:00,NaN,10910200.0,NaN
2026-11-06 14:21:31+07:00,10910200.0,NaN,NaN
2026-11-06 14:21:33+07:00,10910300.0,10910300.0,0.0
2026-11-06 14:21:35+07:00,10910400.0,10910400.0,0.0
2026-11-06 14:21:41+07:00,10910500.0,10910500.0,0.0
2026-11-06 14:21:42+07:00,10911500.0,10911500.0,0.0
2026-11-06 14:21:44+07:00,NaN,10911600.0,NaN
2026-11-06 14:21:45+07:00,10911600.0,NaN,NaN
2026-11-06 14:21:46+07:00,10911700.0,10911700.0,0.0


`kbs` timestamps are often 1s later than `vci` timestamps

In [18]:
fpt[fpt["time"] == "2026-11-06 14:22:56+07:00"]

,time,price,volume,match_type,id,total_so_far,minute
13866,2026-11-06 14:22:56+07:00,50.9,500,Sell,534592561,11000100,2026-11-06 14:22:00+07:00


In [19]:
fpt_kbs[fpt_kbs["time"] == "2026-11-06 14:22:56+07:00"]

,time,price,volume,match_type,id,total_so_far,minute
4701,2026-11-06 14:22:56+07:00,50.9,600,sell,20261106_11000200,11000200,2026-11-06 14:22:00+07:00


in `vci` volume in 14:22:56 is only 500 but in kbs volume in 14:22:56 is 600 \
-> missing 100 volume

14:22:56 is inside page 1 (14:18:31 -> 14:45:00).\
inside page 1 -> page 1 is fetched without `last_id`\
-> the loop can't affect it\
-> not a paging bug, the error comes from VCI's data\
\
Root cause unknown (provider side)

Conclusion:
- The missing volume is a trade at `14:22:56`. `vci` is only 500 but `kbs` is 600
- KBS = OHLCV, so VCI is off by 100
- The root is not caused by the loop but provider side
- OHLCV is the source of truth for daily volume, and the pipeline gets a daily check